# Lenguajes y Autómatas II (SCD-1016)
## Sesión 02: Gestión Dinámica de Memoria, Nodos y Sincronización Web
**Docente:** Mtro. Iván Márquez (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2`

---
### El Ciclo Metodológico de la Sesión
1. **Fase 1 (El Artesano):** Construir y liberar una lista enlazada de tokens a mano (en C o Rust).
2. **Fase 2 (El Científico):** Definir el contrato C-R-E-O para la gestión segura de nodos en memoria y consultar a la IA.
3. **Fase 3 (El Auditor):** Ejecutar el *Test Harness* automatizado con pruebas de estrés de memoria.
4. **Entrega:** Completar `auditorias/auditoria_sesion02.md` y subir commit a tu Fork.

### Paso 0: Verificación del Entorno
Ejecuta esta celda para verificar la disponibilidad de compiladores en la máquina virtual:

In [ ]:
import os

if os.system("which rustc > /dev/null 2>&1") != 0:
    print("⏳ Configurando compilador de Rust...")
    !apt-get update -qq && apt-get install -y -qq rustc > /dev/null 2>&1
    print("✅ Rust instalado con éxito.")

print("\n--- COMPILADORES ACTIVOS ---")
!gcc --version | head -n 1
!rustc --version

---
## SECCIÓN A: TRACK C (Base Oficial)

### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Completa las funciones `crear_token()` y `liberar_lista()` asegurando que no existan fugas de memoria (*memory leaks*) ni accesos posteriores a la liberación (*use-after-free*).

In [ ]:
%%writefile tokens_artesano_c.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct Token {
    char lexema[32];
    int tipo;
    struct Token *siguiente;
} Token;

Token* crear_token(const char *lexema, int tipo) {
    Token *nuevo = (Token*)malloc(sizeof(Token));
    if (nuevo == NULL) return NULL;
    
    strncpy(nuevo->lexema, lexema, sizeof(nuevo->lexema) - 1);
    nuevo->lexema[sizeof(nuevo->lexema) - 1] = '\0';
    nuevo->tipo = tipo;
    nuevo->siguiente = NULL;
    return nuevo;
}

void liberar_lista(Token *cabeza) {
    // TODO (El Artesano): Libera cada nodo usando un apuntador auxiliar
    Token *actual = cabeza;
    while (actual != NULL) {
        Token *aux = actual->siguiente;
        free(actual);
        actual = aux;
    }
}

int main() {
    Token *cabeza = crear_token("while", 100);
    cabeza->siguiente = crear_token("contador", 200);
    cabeza->siguiente->siguiente = crear_token("<", 300);
    
    Token *curr = cabeza;
    while (curr) {
        printf("Token: %-12s | Tipo: %d | Dir: %p\n", curr->lexema, curr->tipo, (void*)curr);
        curr = curr->siguiente;
    }
    
    liberar_lista(cabeza);
    printf("[MEMORIA] Lista liberada con exito.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra tokens_artesano_c.c -o tokens_c && ./tokens_c

### Fase 2: El Científico — Track C
1. Llena la matriz C-R-E-O en `auditorias/auditoria_sesion02.md`.
2. Solicita a la IA una implementación optimizada que incluya inserción al final (`insertar_token_final()`) y liberación segura.
3. Guarda el código devuelto en la siguiente celda:

In [ ]:
%%writefile codigo_ia_tokens_c.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct Token {
    char lexema[32];
    int tipo;
    struct Token *siguiente;
} Token;

// PEGA AQUÍ LAS FUNCIONES GENERADAS POR LA IA
Token* crear_token_ia(const char *lexema, int tipo) {
    if (!lexema) return NULL;
    Token *nuevo = (Token*)malloc(sizeof(Token));
    if (!nuevo) return NULL;
    strncpy(nuevo->lexema, lexema, sizeof(nuevo->lexema) - 1);
    nuevo->lexema[sizeof(nuevo->lexema) - 1] = '\0';
    nuevo->tipo = tipo;
    nuevo->siguiente = NULL;
    return nuevo;
}

void liberar_lista_ia(Token *cabeza) {
    Token *actual = cabeza;
    while (actual) {
        Token *sig = actual->siguiente;
        free(actual);
        actual = sig;
    }
}


### Fase 3: El Auditor — Test Harness en C
Ejecuta la suite de pruebas unitarias sobre la gestión de memoria en C:

In [ ]:
%%writefile test_harness_tokens_c.c
#include <stdio.h>
#include <assert.h>
#include <string.h>

typedef struct Token {
    char lexema[32];
    int tipo;
    struct Token *siguiente;
} Token;

Token* crear_token_ia(const char *lexema, int tipo);
void liberar_lista_ia(Token *cabeza);

int main() {
    printf("=== AUDITORIA CRITICA: MEMORIA Y TOKENS (TRACK C) ===\n");

    // Test 1: Liberación sobre lista vacía (NULL)
    printf("[TEST 1] Liberar cabeza NULL... ");
    liberar_lista_ia(NULL);
    printf("PASO ✅\n");

    // Test 2: Creación con lexema NULL
    printf("[TEST 2] Token con lexema NULL... ");
    Token *nulo = crear_token_ia(NULL, 0);
    assert(nulo == NULL);
    printf("PASO ✅\n");

    // Test 3: Truncamiento seguro de lexema largo (>31 chars)
    printf("[TEST 3] Truncamiento de buffer overflow... ");
    Token *largo = crear_token_ia("identificador_extremadamente_largo_que_supera_limite", 200);
    assert(largo != NULL);
    assert(strlen(largo->lexema) == 31);
    printf("PASO ✅\n");

    // Test 4: Creación y liberación de lista con 3 nodos
    printf("[TEST 4] Cadena de 3 nodos enlazados... ");
    Token *c = crear_token_ia("int", 10);
    c->siguiente = crear_token_ia("x", 20);
    c->siguiente->siguiente = crear_token_ia(";", 30);
    liberar_lista_ia(c);
    liberar_lista_ia(largo);
    printf("PASO ✅\n");

    printf("\nRESULTADO TRACK C: Estructura de memoria blindada con exito.\n");
    return 0;
}


In [ ]:
!gcc -Wall -Wextra codigo_ia_tokens_c.c test_harness_tokens_c.c -o test_suite_tokens_c && ./test_suite_tokens_c

---
## SECCIÓN B: TRACK RUST (Track Alternativo)

### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Modela una lista de tokens utilizando `Option<Box<Token>>` y verifica el comportamiento de liberación automática (RAII).

In [ ]:
%%writefile tokens_artesano_rs.rs
#[derive(Debug)]
pub struct Token {
    pub lexema: String,
    pub tipo: i32,
    pub siguiente: Option<Box<Token>>,
}

impl Token {
    pub fn nuevo(lexema: &str, tipo: i32) -> Self {
        Token {
            lexema: lexema.to_string(),
            tipo,
            siguiente: None,
        }
    }
}

fn main() {
    let mut nodo1 = Token::nuevo("while", 100);
    let mut nodo2 = Token::nuevo("contador", 200);
    let nodo3 = Token::nuevo("<", 300);
    
    nodo2.siguiente = Some(Box::new(nodo3));
    nodo1.siguiente = Some(Box::new(nodo2));
    
    println!("[RUST] Estructura creada:\n{:#?}", nodo1);
}

In [ ]:
!rustc tokens_artesano_rs.rs -o tokens_rs && ./tokens_rs

### Fase 2: El Científico — Track Rust
1. Define la matriz C-R-E-O considerando la propiedad (*ownership*) en Rust.
2. Solicita al LLM la implementación de un método `insertar_final()` para la estructura `Token`.
3. Guarda el código en la siguiente celda:

In [ ]:
%%writefile codigo_ia_tokens_rs.rs
#[derive(Debug, PartialEq)]
pub struct Token {
    pub lexema: String,
    pub tipo: i32,
    pub siguiente: Option<Box<Token>>,
}

impl Token {
    pub fn nuevo(lexema: &str, tipo: i32) -> Self {
        Token {
            lexema: lexema.to_string(),
            tipo,
            siguiente: None,
        }
    }

    pub fn insertar_final(&mut self, nuevo: Token) {
        let mut actual = self;
        while let Some(ref mut sig) = actual.siguiente {
            actual = sig;
        }
        actual.siguiente = Some(Box::new(nuevo));
    }
}


### Fase 3: El Auditor — Test Harness en Rust
Ejecuta la suite de pruebas para verificar el encadenamiento de nodos en Rust:

In [ ]:
%%writefile test_harness_tokens_rs.rs
mod codigo_ia_tokens_rs;
use codigo_ia_tokens_rs::Token;

fn main() {
    println!("=== AUDITORIA CRITICA: GESTION DE NODOS (TRACK RUST) ===");

    // Test 1: Creación individual
    print!("[TEST 1] Creacion de nodo unico... ");
    let mut cabeza = Token::nuevo("let", 10);
    assert_eq!(cabeza.lexema, "let");
    assert_eq!(cabeza.siguiente, None);
    println!("PASO ✅");

    // Test 2: Inserción al final
    print!("[TEST 2] Insercion encadenada al final... ");
    cabeza.insertar_final(Token::nuevo("identificador_y", 20));
    cabeza.insertar_final(Token::nuevo(";", 30));
    assert!(cabeza.siguiente.is_some());
    println!("PASO ✅");

    println!("\nRESULTADO TRACK RUST: Todas las pruebas pasaron con exito.\n");
}


In [ ]:
!rustc test_harness_tokens_rs.rs -o test_suite_tokens_rs && ./test_suite_tokens_rs